# Examples of 関数＋Loop

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import folium

# 日本語フォントの設定（Mac用）
plt.rcParams['font.family'] = 'Hiragino Sans'

# 日本語フォントの設定（PC用）
# plt.rcParams['font.family'] = 'MS Gothic'

# データセットの読み込み
csv_file_path = 'emdat.csv'  
emdat = pd.read_csv(csv_file_path)

# データのプレビュー
emdat.head()


まずは年ごとに災害別の線グラフを作ってくれる関数を作成

In [ ]:
def plot_disaster_counts(data, disaster_type):
    """
    指定した災害タイプの災害件数を年ごとに集計し、折れ線グラフを作成する。

    Parameters:
        data (DataFrame): データセット
        disaster_type (str): 表示する災害の種類
    """
    # データをフィルタリング
    filtered_data = data[data['Disaster Type'] == disaster_type]
    
    # 年ごとに災害件数を集計
    yearly_counts = filtered_data.groupby('Start Year')['Disaster Type'].count()
    
    # グラフを作成
    plt.figure(figsize=(12, 6))
    plt.plot(yearly_counts.index, yearly_counts.values, marker='o', label=f'{disaster_type} 件数')
    plt.title(f'{disaster_type} の年ごとの災害件数の傾向')
    plt.xlabel('年')
    plt.ylabel('災害件数')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# One chart
plot_disaster_counts(emdat,"Storm")

Loopを使って一気に複数のチャートを作成！


In [ ]:
# disaster types
disaster_types = emdat['Disaster Type'].unique()

# 洪水の災害件数を年ごとに集計し、折れ線グラフを作成
for disaster in disaster_types:
    plot_disaster_counts(emdat, disaster)

死者数の棒グラフのトップ１０国

In [ ]:
def plot_total_deaths_bar(data, disaster_type):
    """
    指定した災害タイプで死亡者数が最も多い国トップ10を表示するバーチャートを作成します。

    Parameters:
        data (DataFrame): データセット
        disaster_type (str): 表示する災害の種類
    """
    # データをフィルタリングし、国ごとに集計
    filtered_data = data[data['Disaster Type'] == disaster_type]
    country_deaths = filtered_data.groupby('Country')['Total Deaths'].sum().sort_values(ascending=False).head(10)
    
    # バーチャートを作成
    plt.figure(figsize=(10, 6))
    country_deaths.plot(kind='bar', color='skyblue')
    plt.title(f'Top 10 Countries by Total Deaths for {disaster_type}')
    plt.xlabel('Country')
    plt.ylabel('Total Deaths')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
plot_total_deaths_bar(emdat,"Earthquake")

In [ ]:
# 関数を呼び出す
for disaster in disaster_types:
    plot_total_deaths_bar(emdat, disaster)

In [ ]:
def plot_disaster_map(data, disaster_type):
    """
    指定した災害タイプの地理的分布をFolium地図で表示する。

    Parameters:
        data (DataFrame): データセット
        disaster_type (str): 表示する災害の種類
    """
    # データをフィルタリング
    filtered_data = data[data['Disaster Type'] == disaster_type]
    
    # ベースマップを作成
    disaster_map = folium.Map(location=[20, 0], zoom_start=2)

    # 各災害の位置にマーカーを追加
    for _, row in filtered_data.iterrows():
        if not pd.isna(row['Latitude']) and not pd.isna(row['Longitude']):
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=5,
                color='red',
                fill=True,
                fill_opacity=0.6,
                popup=f"国: {row['Country']}, 死亡者数: {row['Total Deaths']}"
            ).add_to(disaster_map)
    
    return disaster_map

# 関数を呼び出す
plot_disaster_map(emdat, 'Flood')

In [ ]:
# 関数を呼び出す
for disaster in disaster_types:
    plot_disaster_map(emdat, disaster)

In [ ]:
def plot_disaster_by_region(data):
    """
    地域ごとに災害タイプ別の発生件数を集計し、棒グラフや折れ線グラフで比較する。

    Parameters:
        data (DataFrame): データセット
    """
    # 地域ごとに災害タイプ別の発生件数を集計
    disaster_counts = data.groupby(['Region', 'Disaster Type']).size().unstack().fillna(0)
    
    # 棒グラフの作成
    disaster_counts.plot(kind='bar', stacked=True, figsize=(14, 8))
    plt.title('地域ごとの災害タイプ別発生件数（棒グラフ）')
    plt.xlabel('地域')
    plt.ylabel('発生件数')
    plt.legend(title='災害タイプ')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()
    
    # 折れ線グラフの作成
    disaster_counts.plot(kind='line', figsize=(14, 8), marker='o')
    plt.title('地域ごとの災害タイプ別発生件数（折れ線グラフ）')
    plt.xlabel('地域')
    plt.ylabel('発生件数')
    plt.legend(title='災害タイプ')
    plt.grid(True)
    plt.show()

# 関数を呼び出す
plot_disaster_by_region(emdat)

In [ ]:
from folium.plugins import HeatMap

def plot_disaster_heatmap(data):
    """
    年ごとの災害タイプ別発生件数をヒートマップで可視化する。

    Parameters:
        data (DataFrame): データセット
    """
    # データをフィルタリングして、緯度と経度が存在する行のみを使用
    filtered_data = data.dropna(subset=['Latitude', 'Longitude'])
    
    # ヒートマップ用のデータを作成
    heat_data = []
    for year in filtered_data['Start Year'].unique():
        year_data = filtered_data[filtered_data['Start Year'] == year]
        for disaster_type in year_data['Disaster Type'].unique():
            type_data = year_data[year_data['Disaster Type'] == disaster_type]
            for _, row in type_data.iterrows():
                heat_data.append([row['Latitude'], row['Longitude'], 1])
    
    # ベースマップを作成
    heatmap_map = folium.Map(location=[20, 0], zoom_start=2)
    
    # ヒートマップを追加
    HeatMap(heat_data).add_to(heatmap_map)
    
    return heatmap_map

# 関数を呼び出す
plot_disaster_heatmap(emdat)

In [ ]:
def plot_filtered_disaster_map(data, disaster_type, death_threshold):
    """
    指定した災害タイプのうち、死亡者数が閾値以上の災害のみをFolium地図で表示する。

    Parameters:
        data (DataFrame): データセット
        disaster_type (str): 表示する災害の種類
        death_threshold (int): 表示する災害の死亡者数の閾値
    """
    # データをフィルタリング
    filtered_data = data[(data['Disaster Type'] == disaster_type) & (data['Total Deaths'] >= death_threshold)]
    
    # ベースマップを作成
    disaster_map = folium.Map(location=[20, 0], zoom_start=2)

    # 各災害の位置にマーカーを追加
    for _, row in filtered_data.iterrows():
        if not pd.isna(row['Latitude']) and not pd.isna(row['Longitude']):
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=5,
                color='red',
                fill=True,
                fill_opacity=0.6,
                popup=f"国: {row['Country']}, 死亡者数: {row['Total Deaths']}"
            ).add_to(disaster_map)
    
    return disaster_map

# 関数を呼び出す（例: 死亡者数が1000以上の洪水を表示）
plot_filtered_disaster_map(emdat, 'Earthquake', 5000)

In [ ]:
def plot_disasters_by_iso(data, iso_code):
    """
    指定したISOコードの国における災害タイプ別の総死亡者数を表示するバーチャートを作成します。

    Parameters:
        data (DataFrame): データセット
        iso_code (str): 国を識別するISOコード
    """
    # データをフィルタリング
    filtered_data = data[data['ISO'] == iso_code]
    
    # 災害タイプごとに総死亡者数を集計
    disaster_deaths = filtered_data.groupby('Disaster Type')['Total Deaths'].sum().sort_values(ascending=False)
    
    # バーチャートを作成
    plt.figure(figsize=(10, 6))
    disaster_deaths.plot(kind='bar', color='skyblue')
    plt.title(f'Total Deaths by Disaster Type for ISO: {iso_code}')
    plt.xlabel('Disaster Type')
    plt.ylabel('Total Deaths')
    plt.xticks(rotation=45)
    plt.show()

# 関数を呼び出す（例: ISOコードが'USA'の国の災害タイプ別総死亡者数を表示）
plot_disasters_by_iso(emdat, 'USA')

In [ ]:
import matplotlib.pyplot as plt

def plot_disasters_in_iso(data, iso_code):
    """
    指定されたISOコードの災害を地図上にプロットする静的マップを作成します。

    Parameters:
        data (DataFrame): データセット
        iso_code (str): ISOコード (例: 'JPN', 'USA')
    """
    # データをフィルタリング
    filtered_data = data[data['ISO'] == iso_code]

    if filtered_data.empty:
        print(f"指定されたISOコード {iso_code} のデータが見つかりませんでした。")
        return

    # 緯度と経度を取得
    latitudes = filtered_data['Latitude']
    longitudes = filtered_data['Longitude']

    # 国名を取得
    country_name = filtered_data['Country'].iloc[0]

    # プロットの準備
    plt.figure(figsize=(5, 5))
    plt.scatter(longitudes, latitudes, c='red', alpha=0.7, edgecolors='k', s=50)

    # 地図のズームを調整
    plt.title(f'{country_name} の災害マップ ({iso_code})', fontsize=16)
    plt.xlabel('経度', fontsize=12)
    plt.ylabel('緯度', fontsize=12)

    # 軸の範囲を動的に設定
    plt.xlim(longitudes.min() - 1, longitudes.max() + 1)
    plt.ylim(latitudes.min() - 1, latitudes.max() + 1)

    # グリッドとラベル
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# 関数を呼び出す
plot_disasters_in_iso(emdat, 'JPN')

In [ ]:
!pip install cartopy

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_disasters_static_map(data, iso_code):
    """
    指定されたISOコードの災害を静的マップにプロットします。

    Parameters:
        data (DataFrame): データセット
        iso_code (str): ISOコード (例: 'JPN', 'USA')
    """
    # データをフィルタリング
    filtered_data = data[data['ISO'] == iso_code]

    if filtered_data.empty:
        print(f"指定されたISOコード {iso_code} のデータが見つかりませんでした。")
        return

    # 緯度と経度を取得
    latitudes = filtered_data['Latitude']
    longitudes = filtered_data['Longitude']

    # 国名を取得
    country_name = filtered_data['Country'].iloc[0]

    # プロットの準備
    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([longitudes.min() - 1, longitudes.max() + 1, 
                   latitudes.min() - 1, latitudes.max() + 1], crs=ccrs.PlateCarree())

    # 地図に特徴を追加
    ax.add_feature(cfeature.LAND, edgecolor='black', zorder=0)
    ax.add_feature(cfeature.COASTLINE, zorder=1)
    ax.add_feature(cfeature.BORDERS, linestyle=':', zorder=2)
    ax.add_feature(cfeature.LAKES, alpha=0.5, zorder=1)
    ax.add_feature(cfeature.RIVERS, alpha=0.5, zorder=1)

    # 災害のプロット
    ax.scatter(longitudes, latitudes, color='red', edgecolor='black', s=50, label='災害', zorder=3)

    # タイトルと凡例
    plt.title(f'{country_name} の災害マップ ({iso_code})', fontsize=16)
    plt.legend(loc='lower left')
    plt.show()

# 関数を呼び出す
plot_disasters_static_map(emdat, 'JPN')

In [ ]:
# loop through all the countries and plot the disasters
for iso_code in emdat['ISO'].unique():
    # Filter out rows with NaN or Inf in Latitude or Longitude
    filtered_data = emdat[(emdat['ISO'] == iso_code) & emdat['Latitude'].notna() & emdat['Longitude'].notna()]
    if not filtered_data.empty:
        plot_disasters_static_map(filtered_data, iso_code)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import pandas as pd

def plot_3d_line_graph(data):
    """
    Region, Start Year, Total Deathsの3つの項目を使って3D折れ線グラフを作成する。

    Parameters:
        data (DataFrame): データセット
    """
    # データをフィルタリングして必要な列を抽出
    filtered_data = data[['Region', 'Start Year', 'Total Deaths']].dropna()
    
    # ピボットテーブルを作成
    pivot_table = filtered_data.pivot_table(values='Total Deaths', index='Start Year', columns='Region', aggfunc='sum').fillna(0)
    
    # Total Deathsの最大値で地域をソート
    sorted_regions = pivot_table.max().sort_values().index
    pivot_table = pivot_table[sorted_regions]
    
    # 3Dプロットの作成
    fig = plt.figure(figsize=(14, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 各地域のデータをプロット
    for i, region in enumerate(pivot_table.columns):
        ax.plot(pivot_table.index, [i] * len(pivot_table), pivot_table[region], label=region)
    
    ax.set_title('Region, Start Year, Total Deathsの3D折れ線グラフ')
    ax.set_xlabel('Start Year')
    ax.set_ylabel('Region')
    ax.set_zlabel('Total Deaths')
    ax.set_yticks(range(len(pivot_table.columns)))
    ax.set_yticklabels(pivot_table.columns)
    ax.legend()
    
    plt.show()

# 関数を呼び出す
plot_3d_line_graph(emdat)

In [ ]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_disaster_heatmap_binned(data, bin_width=5):
    """
    年ごとの災害タイプ別発生件数をヒートマップで可視化する（指定した年の範囲でグループ化）。
    
    Parameters:
        data (DataFrame): 災害データセット
        bin_width (int): 年を区切る間隔（例: 5なら5年ごと）
    """
    # 年を指定した間隔で区切る
    data['Year Bin'] = (data['Start Year'] // bin_width) * bin_width

    # グループ化して集計
    grouped_data = data.groupby(['Year Bin', 'Disaster Type']).size().unstack(fill_value=0)

    # ヒートマップの描画
    plt.figure(figsize=(18, 12))
    sns.heatmap(
        grouped_data, 
        annot=True, 
        fmt='d', 
        cmap='coolwarm', 
        vmin=0, 
        cbar_kws={'label': '件数'}
    )
    plt.title(f'年ごとの災害タイプ別発生件数（{bin_width}年ごとに区切り）', fontsize=20)
    plt.xlabel('災害タイプ', fontsize=16)
    plt.ylabel('年（区切り）', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ヒートマップを描画（5年ごとに区切る）
plot_disaster_heatmap_binned(emdat, bin_width=2)